# Week 34, Day 5: Rate codes and temporal codes: what spike timing adds

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w034/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Decoding from counts and from latency

Write `nearest_mean(train_x, train_y, test_x)`, which assigns each test value to the stimulus whose training mean is closest. Use the first 100 trials of each stimulus to train and the rest to test, and store the test accuracy of decoding from counts and from latencies in `acc`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2008)
stim = np.repeat(np.arange(4), 200)
latency = 40 - 6 * stim + r.normal(0, 3, stim.size)          # ms
count = r.poisson(2 + 0.3 * stim)                             # spikes in the first 50 ms
train = np.tile(np.r_[np.ones(100, bool), np.zeros(100, bool)], 4)

def nearest_mean(train_x, train_y, test_x):
    classes = np.unique(train_y)
    means = np.array([train_x[train_y == c].mean() for c in classes])
    return ...

acc = {}
for name, x in (("count", count), ("latency", latency)):
    pred = nearest_mean(x[train], stim[train], x[~train])
    acc[name] = np.mean(pred == stim[~train])
    print(f"decoding from {name:7s}: accuracy {acc[name]:.2f} (chance 0.25)")

In [ ]:
import numpy as np
def test_decoder():
    "nearest_mean assigns each test value to the class whose training mean is closest"
    tr_x, tr_y = np.array([0.0, 0.2, 5.0, 5.2]), np.array([0, 0, 1, 1])
    assert np.array_equal(nearest_mean(tr_x, tr_y, np.array([0.1, 4.0])), [0, 1])

def test_accuracy():
    "latency decodes above 0.65 and counts below 0.45, against chance at 0.25"
    assert acc["latency"] > 0.65 and acc["count"] < 0.45

test_decoder()
test_accuracy()
print("All checks passed")

### Exercise: A code that needs no clock

Cell A's latency is 40 - 6s ms and cell B's 22 + 6s ms, each with 3 ms of private jitter, and on each trial both are shifted by the same onset jitter of standard deviation 15 ms. Decode the stimulus from A's absolute latency and from the difference B - A, and store the accuracies in `acc_j`.

In [ ]:
onset = r.normal(0, 15, stim.size)                            # shared, unknown to the decoder
lat_A = 40 - 6 * stim + r.normal(0, 3, stim.size) + onset
lat_B = 22 + 6 * stim + r.normal(0, 3, stim.size) + onset
acc_j = {}
for name, x in (("absolute", lat_A), ("relative", ...)):
    acc_j[name] = ...
    print(f"{name:8s} latency: accuracy {acc_j[name]:.2f}")
plt.scatter(lat_A, lat_B, c=stim, s=6); plt.xlabel("cell A latency (ms)"); plt.ylabel("cell B latency (ms)"); plt.show()

In [ ]:
import numpy as np
def test_jitter():
    "with 15 ms of onset jitter, cell A's absolute latency decodes below 0.55 and the latency difference above 0.75"
    assert acc_j["absolute"] < 0.55 and acc_j["relative"] > 0.75

test_jitter()
print("All checks passed")